# TechArena CTR: CatBoost GPU capacity experiment
Use the original PRIVATE prepared TechArena dataset. GPU required.
Fixed 1600 depth-8 trees and 25% rank blend. No validation-label fitting or early stopping.
The adjusted target is 0.83 and remains unproven until both dates pass.


In [ ]:
from pathlib import Path
import importlib.util, hashlib
import torch
assert torch.cuda.is_available(), "Enable Kaggle GPU"
roots = list(Path("/kaggle/input").rglob("bundle_manifest.json"))
assert len(roots) == 1, "Attach the original prepared TechArena dataset"
DATA_ROOT = roots[0].parent
source_path = Path("/kaggle/working/catboost_gpu_runner.py")
source_path.write_text("\"\"\"Fixed-capacity CatBoost GPU expert on the existing causal feature bundle.\n\nNo evaluation labels enter fitting or early stopping. Main day6 is followed by\nunchanged day5 confirmation only if its conditional gain gate passes.\n\"\"\"\nimport gc\nimport importlib.util\nimport json\nimport time\nfrom pathlib import Path\n\nimport catboost\nimport numpy as np\nfrom catboost import CatBoostClassifier, Pool\n\nPARAMS=dict(iterations=1600,depth=8,learning_rate=.04,l2_leaf_reg=10.,\n            loss_function=\"Logloss\",task_type=\"GPU\",devices=\"0\",border_count=63,\n            bootstrap_type=\"Bernoulli\",subsample=.8,random_seed=6801,\n            boosting_type=\"Plain\",thread_count=4,allow_writing_files=False,\n            verbose=100)\nBLEND=.25\n\n\ndef helpers(root):\n    spec=importlib.util.spec_from_file_location(\"ctr_cb_gpu_helpers\",Path(root)/\"runner.py\")\n    module=importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef predict(model,features):\n    out=[]\n    for off in range(0,len(features),25000):\n        x=features.block(off,min(off+25000,len(features)))\n        out.append(model.predict_proba(x,thread_count=4)[:,1])\n    return np.concatenate(out)\n\n\ndef fit(root,output,manifest,day,h):\n    started=time.monotonic()\n    output.mkdir(parents=True,exist_ok=False)\n    days=list(range(day-3,day))\n    validation=manifest[\"validation\"][str(day)]\n    specs=[manifest[\"train\"][str(d)] for d in days]\n    labels=[]\n    for spec in specs:\n        meta=json.loads((root/spec[\"metadata\"]).read_text())\n        assert meta[\"last_timestamp\"]<validation[\"cutoff\"]\n        labels.append(np.load(root/spec[\"labels\"]))\n    y=np.concatenate(labels).astype(\"float32\")\n    feature=h.FeatureBlocks(root,[s[\"features\"] for s in specs])\n    assert len(feature)==len(y)\n    recipe=dict(parameters=PARAMS,blend_weight=BLEND,day=day,fit_days=days,\n                fit_rows=len(y),columns=354,validation_cutoff=validation[\"cutoff\"],\n                input_manifest_sha256=h.digest(root/\"bundle_manifest.json\"),\n                source_sha256=h.digest(Path(__file__)),catboost=catboost.__version__,\n                selection=\"fixed before scoring; no eval_set or early stopping\")\n    (output/\"recipe.json\").write_text(json.dumps(recipe,indent=2))\n    temporary=output/\"fit_features.npy\"\n    assert temporary.resolve().parent==output.resolve()\n    matrix=np.lib.format.open_memmap(temporary,mode=\"w+\",dtype=\"float32\",shape=(len(y),354))\n    for off in range(0,len(y),50000):\n        end=min(off+50000,len(y));matrix[off:end]=feature.block(off,end)\n        if off%500000==0:print(dict(stage=\"prepare_features\",day=day,rows=end,total=len(y)),flush=True)\n    matrix.flush()\n    pool=Pool(matrix,label=y,thread_count=4)\n    model=CatBoostClassifier(**PARAMS)\n    print(dict(stage=\"catboost_gpu_fit\",day=day,rows=len(y),iterations=PARAMS[\"iterations\"]),flush=True)\n    model.fit(pool)\n    model_path=output/\"model.cbm\"\n    model.save_model(str(model_path))\n    del pool\n    matrix._mmap.close()\n    del matrix,y\n    gc.collect()\n    # Remove only the scratch array authored by this fresh experiment.\n    assert temporary.resolve().parent==output.resolve()\n    temporary.unlink()\n    with np.load(root/validation[\"data\"]) as z:\n        val={k:z[k].copy() for k in z.files}\n    xv=h.FeatureBlocks(root,[validation[\"features\"]])\n    assert len(xv)==len(val[\"y\"]) and len(xv)%10==0\n    raw=predict(model,xv)\n    assert np.isfinite(raw).all() and ((raw>=0)&(raw<=1)).all()\n    pred=h.rank((1-BLEND)*h.rank(val[\"baseline\"])+BLEND*h.rank(raw))\n    m=h.metrics(val[\"y\"],pred,val[\"baseline\"],val[\"weights\"])\n    restored=CatBoostClassifier()\n    restored.load_model(str(model_path))\n    replay=predict(restored,xv)\n    replay_error=float(np.max(np.abs(replay-raw)))\n    assert replay_error<1e-12,replay_error\n    m.update(day=day,raw_auc=float(h.roc_auc_score(val[\"y\"],raw)),rows=len(raw),\n             saved_model_replay_max_error=replay_error,saved_checkpoint_replay=\"PASS\",\n             model_sha256=h.digest(model_path),elapsed_seconds=round(time.monotonic()-started,1),\n             official_auc=None,csv_ready=False)\n    np.savez_compressed(output/\"predictions.npz\",raw=raw,pred=pred,**val)\n    (output/\"report.json\").write_text(json.dumps(m,indent=2))\n    print(json.dumps(m,indent=2),flush=True)\n    return m\n\n\ndef run(root,output):\n    root,output=Path(root),Path(output)\n    h=helpers(root)\n    manifest=h.load_bundle(root,verify=True)\n    output.mkdir(parents=True,exist_ok=False)\n    (output/\"catboost_gpu_runner.py\").write_text(Path(__file__).read_text())\n    reports=[fit(root,output/\"day6\",manifest,6,h)]\n    if reports[0][\"positive_gate\"]:\n        reports.append(fit(root,output/\"day5\",manifest,5,h))\n    eligible=len(reports)==2 and all(m[\"positive_gate\"] for m in reports)\n    summary=dict(reports=reports,eligible_for_review=eligible,target=.83,\n                 target_achieved=bool(eligible and reports[0][\"history_adjusted_auc\"]>=.83),\n                 csv_ready=False,official_auc=None,\n                 note=\"Same reused main dates; fixed CatBoost GPU expert. Local metric, not official forecast.\")\n    (output/\"summary.json\").write_text(json.dumps(summary,indent=2))\n    return summary\n", encoding="utf-8")
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == "f95db97cc1ca7f086c49c9ed7393cc2a40ef58ca2fdb6e5274b0b6b85d52c154"
spec = importlib.util.spec_from_file_location("ctr_cb_gpu", source_path)
cb_gpu = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cb_gpu)
RUN_NAME = "ctr-catboost-gpu-001"
output = Path("/kaggle/working") / RUN_NAME
print("CatBoost version:", cb_gpu.catboost.__version__)
print("Frozen GPU parameters:", cb_gpu.PARAMS)
summary = cb_gpu.run(DATA_ROOT, output)
summary


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json, zipfile
print("Experiment:", RUN_NAME)
report = json.loads((output / "summary.json").read_text())
for result in report["reports"]:
    print("Day", result["day"], "Adjusted AUC:", round(result["history_adjusted_auc"], 9), "Ordinary AUC:", round(result["ordinary_auc"], 9), "Gate:", result["positive_gate"])
print("Target 0.83 reached:", report["target_achieved"])
archive = Path("/kaggle/working") / (RUN_NAME + "-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in output.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(output))
display(FileLink(str(archive)))
